### 第一个实验：让 Graph 停下来问人

我们创建一个非常简单的 Graph

```
    START
      │
      ▼
    ask_human
      │
      │ interrupt()
      │
      ▼
    Human
      │
      │ resume
      ▼
    finish
      │
      ▼
     END
```

In [3]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from typing import TypedDict

from langgraph.types import interrupt


class State(TypedDict):
    question: str
    answer: str


def ask_question(state: State):
    answer = interrupt(
        {
            "question": "你是否同意继续",
            "original_question": state["question"],
        }
    )

    return answer

def finish(state: State):
    return {
        "answer": "Human said: " + state["answer"],
    }

builder = StateGraph(State)
builder.add_node("ask_question", ask_question)
builder.add_node("finish", finish)
builder.add_edge(START, "ask_question")
builder.add_edge("ask_question", "finish")
builder.add_edge("finish", END)

checkpoint = InMemorySaver()
graph = builder.compile(checkpointer=checkpoint)

config = {
    "configurable": {
        "thread_id": "hitl-demo-001"
    }
}

result = graph.invoke(
    {
        "question": "我准备发送这封邮件，可以吗？"
    },
    config=config
)

print(result)

{'question': '我准备发送这封邮件，可以吗？', '__interrupt__': [Interrupt(value={'question': '你是否同意继续', 'original_question': '我准备发送这封邮件，可以吗？'}, id='c4e9de54c59e61798826827b7f58042a')]}
